# 🎯 Module 02: Data Preprocessing

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 02 — Data Preprocessing  
> **Date:** 03 January 2026  
> **Previous:** 01 — ML Fundamentals  
> **Next:** 03 — Regression

---

Real-world datasets are **messy**. Before any model can learn from them, we must **preprocess** them:

- Handle **missing values**
- **Encode** categorical variables
- **Scale** numerical features
- Split **X / y** and **train / test**
- Validate with **cross-validation**
- Avoid **data leakage**
- Bundle it all into a **Pipeline**

Good preprocessing is often the difference between a mediocre model and an excellent one.

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Diagnose common data-quality issues
- Impute missing values (mean / median / mode)
- Apply **One-Hot**, **Label**, and **Target** encoding
- Explain when each encoding is appropriate
- Use **StandardScaler**, **MinMaxScaler**, **RobustScaler**
- Explain the **`fit_transform()` vs `transform()`** rule
- Perform **cross-validation**
- Recognize and prevent **data leakage**
- Build a **scikit-learn Pipeline**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

np.random.seed(42)
pd.set_option('display.max_columns', None)

print('✅ Setup ready.')

---
## 🧠 1. What is Data Preprocessing?

Real-world datasets are rarely ready for ML. They may contain:

- Missing values
- Text categories
- Different numerical scales
- Duplicate records
- Incorrect data types
- Irrelevant columns
- Outliers

**Data preprocessing** prepares this data before it's given to a model:

```text
Raw Data
   ↓
Clean Data
   ↓
Transform Features
   ↓
Train/Test Split
   ↓
ML Model
```

> **Good preprocessing can significantly affect model performance.**

---
## 🧹 2. Handling Missing Values

Example:

| Age | Salary | City |
| --: | -----: | --- |
| 23 | 50000 | Islamabad |
| NaN | 65000 | Lahore |
| 29 | NaN | Karachi |

### Numerical Data

**Mean imputation**
```python
df['age'] = df['age'].fillna(df['age'].mean())
```

**Median imputation**
```python
df['age'] = df['age'].fillna(df['age'].median())
```

> Median is often better when the data has **strong outliers**.

### Categorical Data

Use the **most frequent** category:
```python
df['city'] = df['city'].fillna(df['city'].mode()[0])
```

### When to Drop

If a column has extremely large amounts of missing data, you may consider removing it — **but understand why first**.

In [ ]:
# Build a small dataset with missing values
df = pd.DataFrame({
    'age':    [23, np.nan, 29, 35, np.nan, 41, 27, 33],
    'salary': [50000, 65000, np.nan, 80000, 72000, np.nan, 55000, 90000],
    'city':   ['Islamabad', 'Lahore', 'Karachi', np.nan, 'Lahore', 'Karachi', np.nan, 'Islamabad'],
})

print('Raw data:')
print(df)
print('\nMissing per column:')
print(df.isnull().sum())

In [ ]:
# Impute missing values
df_clean = df.copy()

# Numerical → median (robust to outliers)
df_clean['age']    = df_clean['age'].fillna(df_clean['age'].median())
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].median())

# Categorical → mode (most frequent)
df_clean['city'] = df_clean['city'].fillna(df_clean['city'].mode()[0])

print('After imputation:')
print(df_clean)
print('\nRemaining missing values:', df_clean.isnull().sum().sum())

---
## 🔤 3. Encoding Categorical Data

ML algorithms generally require **numerical inputs**. Categories like:

```text
Islamabad, Lahore, Karachi
```

must be converted into numbers.

### One-Hot Encoding

Creates separate binary columns:

| City | Islamabad | Lahore | Karachi |
| --- | ---: | ---: | ---: |
| Islamabad | 1 | 0 | 0 |
| Lahore | 0 | 1 | 0 |
| Karachi | 0 | 0 | 1 |

Best for **nominal categories** where there is **no natural order**.

### Label Encoding

Converts categories into integer labels:

```text
Cat → 0
Dog → 1
Horse → 2
```

⚠️ **Warning:** Some models interpret these integers as **ordered**. Use this carefully for **input features**. It's fine for **target labels**.

### Target Encoding

Replaces each category with a **statistic from the target**:

| City | Average Price |
| --- | ---: |
| Islamabad | 15M |
| Lahore | 12M |
| Karachi | 10M |

Useful for **high-cardinality** features.

⚠️ **Target encoding can cause data leakage** if computed using validation/test data.

In [ ]:
# One-Hot Encoding with pandas (simple)
df_ohe = pd.get_dummies(df_clean, columns=['city'], prefix='city')
print('After One-Hot Encoding:')
print(df_ohe)

In [ ]:
# One-Hot Encoding with scikit-learn (production-friendly)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = ohe.fit_transform(df_clean[['city']])

df_sklearn_ohe = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['city']))
print('Scikit-learn OneHotEncoder output:')
print(df_sklearn_ohe)

In [ ]:
# Label Encoding (useful for target labels)
le = LabelEncoder()
cities = ['Islamabad', 'Lahore', 'Karachi', 'Lahore', 'Karachi']
encoded_labels = le.fit_transform(cities)

print('Original :', cities)
print('Encoded  :', encoded_labels.tolist())
print('Classes  :', le.classes_.tolist())

---
## 📏 4. Feature Scaling

Different features can live on **very different scales**:

```text
Age    = 25
Salary = 150000
```

Some algorithms are **sensitive to scale**:

- KNN
- K-Means
- SVM
- Logistic Regression
- Neural networks

### Standardization (Z-score)

Transforms data to approximately **mean = 0, std = 1**:

$$z = \frac{x - \mu}{\sigma}$$

### Min-Max Scaling

Usually converts to **range [0, 1]**:

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

### Robust Scaling

Uses **median and IQR** — good when there are strong outliers.

### 🚨 Critical rule

```text
Training data → fit_transform()
Test data     → transform()
```

**Do NOT fit the scaler separately on test data.**

In [ ]:
# Demonstrate scaling
np.random.seed(42)
data = np.array([[25, 50000],
                 [30, 80000],
                 [45, 120000],
                 [22, 45000],
                 [38, 95000]], dtype=float)

print('Raw data:')
print(pd.DataFrame(data, columns=['age', 'salary']).round(1))

# StandardScaler
std_scaler = StandardScaler()
data_std = std_scaler.fit_transform(data)
print('\nAfter StandardScaler (mean~0, std~1):')
print(pd.DataFrame(data_std, columns=['age', 'salary']).round(3))

# MinMaxScaler
mm_scaler = MinMaxScaler()
data_mm = mm_scaler.fit_transform(data)
print('\nAfter MinMaxScaler (range 0→1):')
print(pd.DataFrame(data_mm, columns=['age', 'salary']).round(3))

# RobustScaler
rob_scaler = RobustScaler()
data_rob = rob_scaler.fit_transform(data)
print('\nAfter RobustScaler (median=0, IQR-based):')
print(pd.DataFrame(data_rob, columns=['age', 'salary']).round(3))

In [ ]:
# The fit_transform() / transform() rule — demonstrated correctly
X = np.array([[1, 100], [2, 200], [3, 300], [4, 400], [5, 500]], dtype=float)
y = np.array([0, 0, 1, 1, 1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42
)

scaler = StandardScaler()

# ✅ CORRECT
X_train_scaled = scaler.fit_transform(X_train)   # learn μ,σ from TRAIN
X_test_scaled  = scaler.transform(X_test)        # apply SAME μ,σ to TEST

print('Train scaled mean:', X_train_scaled.mean(axis=0).round(6))
print('Train scaled std :', X_train_scaled.std(axis=0).round(6))
print('Test  scaled (uses train stats, so not necessarily mean 0):')
print(X_test_scaled.round(3))

---
## ✂️ 5. Feature Splitting

Separate **features** from **target**:

```python
X = df.drop('price', axis=1)
y = df['price']
```

Then split:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

Common split: **80% train / 20% test**.

---
## 🔄 6. Cross-Validation

Relying on a **single** train/test split is fragile. **Cross-validation** evaluates a model across multiple splits.

### 5-Fold Cross-Validation

```text
Dataset
────────────────────────
Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5
```

Each fold becomes the validation set **once**:

```text
Round 1 → Test Fold 1
Round 2 → Test Fold 2
Round 3 → Test Fold 3
Round 4 → Test Fold 4
Round 5 → Test Fold 5
```

Then we average the scores — a **more reliable estimate** of performance.

In [ ]:
# Cross-validation demo
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

model = LogisticRegression(max_iter=200, random_state=42)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('5-fold CV accuracies:', scores.round(4))
print(f'Mean:   {scores.mean():.4f}')
print(f'Std:    {scores.std():.4f}')

---
## 🚨 7. Data Leakage

**Data leakage** happens when information that should be **unavailable during training** enters the model-building process.

### ❌ Wrong

```text
Fit scaler on ENTIRE dataset
        ↓
Train + Test information leaked
```

### ✅ Correct

```text
Training Data
     ↓
fit scaler
     ↓
transform training

Test Data
     ↓
transform only
```

The same principle applies to:

- Imputation
- Feature selection
- Target encoding
- Scaling
- Feature engineering

> **Rule of thumb:** anything that "learns from data" must see **only the training set**.

In [ ]:
# Illustrate leakage with a synthetic example
np.random.seed(42)
n = 200
X_leak = np.random.randn(n, 3)
y_leak = (X_leak[:, 0] + 0.5 * X_leak[:, 1] > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42
)

# ❌ LEAKY approach
scaler_leak = StandardScaler().fit(X_leak)   # sees test data!
X_tr_leak = scaler_leak.transform(X_tr)
X_te_leak = scaler_leak.transform(X_te)

# ✅ CORRECT approach
scaler_ok = StandardScaler().fit(X_tr)
X_tr_ok = scaler_ok.transform(X_tr)
X_te_ok = scaler_ok.transform(X_te)

print('Leaky scaler mean   :', scaler_leak.mean_.round(4))
print('Correct scaler mean :', scaler_ok.mean_.round(4))
print('\nDifference may look small here, but on real data the leak can inflate scores significantly.')

---
## 🏗️ 8. Preprocessing Pipeline

Scikit-learn lets us combine preprocessing **and** modeling into a single object:

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
```

### Why pipelines matter

- Preprocessing is applied **consistently**
- No leakage — the pipeline handles `fit`/`transform` correctly
- Cross-validation "just works"
- Easy to deploy as one artifact

> We'll go deeper into `Pipeline` and `ColumnTransformer` in **Module 14**.

In [ ]:
# Full pipeline: scaling + logistic regression
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=200, random_state=42)),
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)

from sklearn.metrics import accuracy_score
print(f'Test accuracy (pipeline): {accuracy_score(y_test, preds):.4f}')

In [ ]:
# Cross-validate the pipeline directly — no leakage by construction
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print('Pipeline 5-fold CV:', cv_scores.round(4))
print(f'Mean accuracy: {cv_scores.mean():.4f}')

---
## 🧠 9. End-to-End Preprocessing Flow

```text
Raw Dataset
     ↓
Handle missing values
     ↓
Encode City
     ↓
Split X / y
     ↓
Train/Test Split
     ↓
Scale numerical features
     ↓
Train Model
     ↓
Evaluate
```

In [ ]:
# Build a full end-to-end preprocessing demo
df_full = pd.DataFrame({
    'age':    [23, np.nan, 29, 35, 42, np.nan, 27, 33, 51, 46],
    'salary': [50000, 65000, np.nan, 80000, 120000, 72000, np.nan, 55000, 150000, 95000],
    'city':   ['Islamabad', 'Lahore', 'Karachi', np.nan, 'Lahore',
               'Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Lahore'],
    'purchased': [0, 1, 0, 1, 1, 0, 0, 1, 1, 1],
})

print('Raw:')
print(df_full)

# 1. Impute
df_full['age']    = df_full['age'].fillna(df_full['age'].median())
df_full['salary'] = df_full['salary'].fillna(df_full['salary'].median())
df_full['city']   = df_full['city'].fillna(df_full['city'].mode()[0])

# 2. Encode
df_full = pd.get_dummies(df_full, columns=['city'], prefix='city')

print('\nAfter imputation + encoding:')
print(df_full)

In [ ]:
# 3. Split, scale, model
X = df_full.drop('purchased', axis=1)
y = df_full['purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train_scaled, y_train)

train_acc = model.score(X_train_scaled, y_train)
test_acc  = model.score(X_test_scaled,  y_test)

print(f'Train accuracy: {train_acc:.3f}')
print(f'Test  accuracy: {test_acc:.3f}')
print(f'\nGap: {train_acc - test_acc:+.3f}   (small gap → good generalization)')

---
## 📝 10. Practice Checklist

Practice on a small dataset like **Titanic** or any tabular dataset:

- [ ] Find missing values
- [ ] Fill numerical missing values
- [ ] Fill categorical missing values
- [ ] One-hot encode categories
- [ ] Split X and y
- [ ] Create train/test sets
- [ ] Apply `StandardScaler`
- [ ] Apply cross-validation
- [ ] Identify possible leakage

---
## 🏋️ 11. Hands-On Exercises

### Exercise 1 — Compare Imputation Strategies
Create a DataFrame with a numeric column containing outliers + missing values. Impute with `mean`, then `median`. Which is more representative of the data?

### Exercise 2 — When One-Hot Fails
Create a column with **1000 unique categories** and try One-Hot Encoding. How many columns result? Would **target encoding** be better here?

### Exercise 3 — Scaling Comparison
Apply `StandardScaler`, `MinMaxScaler`, and `RobustScaler` to the same data with outliers. Compare the resulting distributions.

### Exercise 4 — Force Data Leakage
Fit a scaler on the **full dataset** (train + test). Then fit correctly on train only. Compare test set accuracy. Does the leaky version look artificially better?

### Exercise 5 — Cross-Validation vs Single Split
Evaluate a classifier with `train_test_split` and then with 5-fold `cross_val_score`. Are the numbers consistent? Which gives a more reliable estimate?

### Exercise 6 — Full Pipeline
Wrap imputation, encoding, and modeling into a `Pipeline`. Verify no leakage by cross-validating the entire pipeline.

---
## 🎤 12. Interview Questions

1. **Why is preprocessing necessary?**  
   Raw data is messy and models require clean, numerical, well-scaled inputs.

2. **Mean vs median imputation?**  
   Median is more robust to outliers; mean is sensitive to extreme values.

3. **What is one-hot encoding?**  
   Creating a binary column for each category.

4. **Label vs One-Hot encoding?**  
   Label encoding assigns an integer per class (may imply false order); one-hot avoids this.

5. **What is target encoding?**  
   Replacing a category with a statistic computed from the target; useful for high-cardinality features.

6. **Why do we scale features?**  
   Some algorithms are sensitive to feature scale (KNN, SVM, K-Means, Logistic Regression, NNs).

7. **StandardScaler vs MinMaxScaler?**  
   Standard → mean 0, std 1. MinMax → range [0,1]. Robust → median/IQR.

8. **When would you use RobustScaler?**  
   When the data contains strong outliers.

9. **What is cross-validation?**  
   A technique that evaluates a model on multiple train/val splits to give a more reliable score.

10. **What is data leakage?**  
    When information from outside the training set leaks into the model during training, inflating scores.

11. **Why `fit_transform()` on train but only `transform()` on test?**  
    Because we must learn statistics (μ, σ, min/max) from train only, then apply them to test.

12. **Why are ML pipelines useful?**  
    They bundle preprocessing + modeling, prevent leakage, and make code reproducible and deployable.

---
## 🏁 13. Key Takeaways

```text
Raw Data
   ↓
Missing Values → Handle
   ↓
Categorical Data → Encode
   ↓
Numerical Features → Scale when needed
   ↓
X / y Split
   ↓
Train / Test Split
   ↓
Cross-Validation
   ↓
Model
```

### 🧭 The Golden Rule

> **Preprocessing must be learned from TRAINING data — never from the test set.**

---

### 🔗 What's Next?

**Module 03 — Regression**

We'll apply everything we've learned to predict **continuous values** — from simple linear regression to Ridge, Lasso, and Polynomial models.